# 03 — Model Training

End-to-end training of a disease module, from a fused panel to an explained,
actionable forecast. This is the notebook that mirrors what
`scripts/run_backtest.py` and the retraining scheduler do in production.

**What is worth watching:**

* the target is shifted **forward** by the forecast horizon — this is a
  forecaster, not a nowcaster, and nothing available only at outbreak time is
  allowed into `X`;
* prediction intervals come from **out-of-sample** residuals (an earlier version
  used in-sample residuals and covered 37% of outcomes at a nominal 95% — see
  section 6);
* every prediction is constructed together with its explanation, so a
  `PredictionResult` without SHAP drivers cannot come out of this path
  (critical rule #2).

In [ ]:
# Make the repo importable regardless of where Jupyter was launched from.
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src").is_dir() and (p / "config").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import logging
logging.getLogger("afya").setLevel(logging.WARNING)   # keep notebook output readable

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print(f"repo root: {ROOT}")

In [ ]:
# Plotting is optional throughout these notebooks: matplotlib is not a hard
# dependency of AFYA-PREDICT, because the platform must install on low-spec
# district hardware. Every notebook falls back to printed tables without it.
#
# Backend selection matters more than it looks. Inside a Jupyter kernel,
# matplotlib configures its own inline backend and we leave it alone. Anywhere
# else - `nbconvert --execute`, CI, a headless server - a GUI backend will block
# forever on a window that never opens (a set-but-unreachable $DISPLAY is enough
# to trigger it), so we force the non-interactive Agg backend.
import os
import sys

try:
    import matplotlib
    _in_kernel = "ipykernel" in sys.modules
    if not os.environ.get("MPLBACKEND") and not _in_kernel:
        matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams["figure.figsize"] = (11, 4)
    plt.rcParams["axes.grid"] = True
    plt.rcParams["grid.alpha"] = 0.3
    HAS_PLT = True
    print(f"matplotlib {matplotlib.__version__} on the "
          f"{matplotlib.get_backend()} backend")
except ImportError:
    HAS_PLT = False
    print("matplotlib not installed - tables will be printed instead of plotted")

In [ ]:
from src.core.config_loader import load_region_config, load_disease_config
from src.core.geo import subset_region

FULL_REGION = load_region_config("tanzania")
print(f"{len(FULL_REGION.districts)} councils across "
      f"{len({d.region for d in FULL_REGION.districts})} regions")

# A small, ecologically diverse subset keeps these notebooks fast to run.
# Swap in FULL_REGION for a national analysis (much slower).
STUDY_DISTRICTS = [
    "Kinondoni",     # dense coastal city
    "Ilala",         # dense coastal city, adjacent to Kinondoni
    "Mwanza City",   # lakeside city
    "Sengerema",     # rural lakeside, low WASH coverage
    "Dodoma City",   # semi-arid central
    "Songea MC",     # southern highlands
]
REGION = subset_region(FULL_REGION, STUDY_DISTRICTS)
pd.DataFrame([d.model_dump() for d in REGION.districts]).set_index("name")

## 1. Which compute backend is available?

The platform prefers XGBoost, then LightGBM, then scikit-learn, and finally
falls back to a gradient-boosted regression tree bundled with the project
(`src/models/gbm.py`) that needs nothing but NumPy.

That fallback is not a stub — it is a real histogram GBM with binning,
depth-limited best-first splits, L2 regularisation, shrinkage, row/column
subsampling and native missing-value handling. It exists because installing a
57 MB wheel over an intermittent district link is not always realistic
(shortcoming #9), and because the offline edge deployment must train locally.

In [ ]:
from src.models.backends import backend_report, build_regressor

report = backend_report()
for key, value in report.items():
    print(f"  {key:20} {value}")

_, info = build_regressor("xgboost")   # what the malaria config asks for
print(f"\nconfig asks for 'xgboost' -> resolved to: {info}")

## 2. Ingest the training panel

Five years of history: enough for the seasonal component and a walk-forward split.

In [ ]:
from src.data_ingestion.normalizer import ingest
from src.models.registry import build_module, list_modules

print("registered disease modules:", ", ".join(list_modules()))

DISEASE = "malaria"
module = build_module(DISEASE, region=REGION)
print(f"\nmodule class : {type(module).__name__}")
print(f"horizon      : {module.horizon} weeks")
print(f"target column: {module.target_column}")
print(f"sources      : {', '.join(module.config.required_sources)}")

SOURCES = sorted(set(module.config.required_sources) | {"dhis2", "livestock_disease"})
panel = ingest(SOURCES, "2019-W01", "2024-W52", region=REGION)
print(f"\npanel: {panel.summary()['weeks']} weeks x {panel.summary()['districts']} districts, "
      f"{panel.source_count()} sources fused")

## 3. Build the design matrix

`FeatureBuilder` fits the lags, materialises them, then layers on rolling
statistics, anomalies, seasonality, spatial context, mobility, interactions and
One Health signals — recording provenance for every column.

In [ ]:
matrix = module.build_feature_matrix(panel)

print(f"X: {matrix.X.shape[0]} district-weeks x {matrix.X.shape[1]} features")
print(f"y: {matrix.y.notna().sum()} non-null targets "
      f"(the final {module.horizon} weeks have no known future)\n")

families = pd.Series(
    [matrix.describe_feature(f)["kind"] for f in matrix.feature_names]
).value_counts().rename("features")
display(families.to_frame())

### Leakage check

The single most common way a forecasting model flatters itself is by seeing the
answer. Two invariants are asserted below rather than assumed.

In [ ]:
target = module.target_column

assert target not in matrix.feature_names, "contemporaneous target leaked into X"
leaks = [f for f in matrix.feature_names if f.startswith(target)]
assert all(any(tag in f for tag in ("lag", "roll", "trend", "yoy", "nbr")) for f in leaks), \
    "unlagged target-derived feature present"
print(f"OK  contemporaneous '{target}' is not a feature")
print(f"OK  {len(leaks)} target-derived features, all explicitly lagged/rolled:")
for f in leaks:
    print(f"      {f}")

# And the target really is the future.
district = matrix.districts[0]
y_local = matrix.y.xs(district, level="district")
cases = panel.values()[target].xs(district, level="district")
weeks = list(cases.index)
print(f"\nHorizon check for {district}:")
print(f"  y at {weeks[0]} = {y_local.loc[weeks[0]]}")
print(f"  cases at {weeks[module.horizon]} = {cases.loc[weeks[module.horizon]]}  <- same value")
print(f"  last {module.horizon} weeks of y are NaN: "
      f"{bool(y_local.iloc[-module.horizon:].isna().all())}")

## 4. Train

`fit_models` fits a **pooled** model across all districts, then a **local** model
for each district with enough history. Districts below the threshold keep the
pooled model — borrowing strength rather than pretending to a local fit that the
data cannot support (shortcoming #8).

Two details that matter:

* rows are weighted by their input quality, so a week built largely from imputed
  drivers contributes less to the fit (critical rule #7);
* residual quantiles for the prediction intervals are estimated on a
  chronological internal holdout, **split by week** so a district's own week
  cannot leak through its neighbours.

In [ ]:
import time

started = time.perf_counter()
module.train(matrix)
elapsed = time.perf_counter() - started

scopes = pd.DataFrame([{
    "scope": scope,
    "rows": m.n_rows,
    "backend": m.backend.resolved,
    "train_weeks": f"{m.train_weeks[0]} .. {m.train_weeks[1]}",
    "in_sample_r2": round(m.train_score, 3),
    "residual_std": round(m.residual_std, 2),
    "ci_lower_q": round(m.residual_quantiles.get("q025", float("nan")), 1),
    "ci_upper_q": round(m.residual_quantiles.get("q975", float("nan")), 1),
} for scope, m in module.models.items()]).set_index("scope")

print(f"trained {len(module.models)} model(s) in {elapsed:.1f}s\n")
display(scopes)
print("\nNote the residual quantiles are wider than in-sample error would suggest -")
print("they come from held-out weeks, which is why the intervals in section 6 hold up.")

## 5. Which signals is the model actually using?

Native importance first (fast), then SHAP (exact, and attributable to a
mechanism). The **source**-level rollup is the one to check: if a single source
dominates, the platform has quietly recreated the Google Flu Trends single-source
failure mode (shortcoming #2).

In [ ]:
from src.explainability.feature_importance import (
    concentration_index, dominant_source_warning, proxy_importance, source_importance,
)
from src.explainability.shap_explainer import ShapExplainer

pooled = module.models["pooled"]
background = matrix.X.dropna(how="all")
sample = background.tail(150)

explainer = ShapExplainer(pooled, background=background, provenance=matrix.provenance)
explanation = explainer.explain(sample)
print(f"SHAP method: {explanation.method}")

# Local accuracy is what makes a "contribution share" meaningful.
predicted = pooled.predict(sample)
error = np.abs(explanation.base_value + explanation.values.sum(axis=1) - predicted).max()
print(f"local accuracy (max |base + sum(shap) - prediction|): {error:.2e}\n")

print("=== by data source ===")
display(source_importance(explanation))
print(f"concentration index (1.0 = single source): {concentration_index(explanation):.3f}")
warning = dominant_source_warning(explanation)
print(warning if warning else "no single source dominates - fusion is working\n")

print("=== by digital proxy ===")
display(proxy_importance(explanation).head(12))

In [ ]:
if HAS_PLT:
    top = proxy_importance(explanation).head(10).iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.barh(top["proxy"], top["share"])
    ax.set_xlabel("share of total |SHAP|")
    ax.set_title(f"{module.config.name}: proxy importance")
    plt.tight_layout(); plt.show()

## 6. Forecast, with everything attached

`predict()` returns a `PredictionResult` carrying the point estimate, a 95%
interval, SHAP drivers with their mechanisms, a natural-language explanation, a
counterfactual, importation risk with named source districts, and costed
recommendations.

In [ ]:
DISTRICT = "Sengerema"
results = module.predict(matrix, DISTRICT, panel=panel)
prediction = results[-1]

print(f"{prediction.disease} | {prediction.district} ({prediction.region})")
print(f"target week      : {prediction.target_week}  "
      f"({module.horizon} weeks of lead time)")
print(f"predicted cases  : {prediction.predicted_cases:,.0f}")
print(f"95% interval     : {prediction.confidence_interval_lower:,.0f} "
      f"- {prediction.confidence_interval_upper:,.0f}")
print(f"risk level       : {prediction.risk_level.upper()}  "
      f"(score {prediction.risk_score:.2f})")
print(f"importation risk : {prediction.importation_risk:.0%}")
print(f"model version    : {prediction.model_version}")

### The explanation — the part that decides whether anyone acts

In [ ]:
import textwrap

print(textwrap.fill(prediction.natural_language_explanation, 100))
print()
if prediction.counterfactual:
    print("COUNTERFACTUAL")
    print(textwrap.fill(prediction.counterfactual, 100))

In [ ]:
drivers = pd.DataFrame([{
    "feature": d.feature,
    "proxy": d.proxy,
    "lag_wk": d.lag_weeks,
    "value": None if d.value is None else round(d.value, 2),
    "shap": round(d.shap_value, 3),
    "share": f"{d.contribution_share:.1%}",
    "direction": d.direction,
    "mechanism": (d.mechanism[:60] + "...") if len(d.mechanism) > 60 else d.mechanism,
} for d in prediction.top_drivers])
display(drivers)

### Where the risk is arriving from (shortcoming #10)

In [ ]:
if prediction.source_districts:
    display(pd.DataFrame([s.model_dump() for s in prediction.source_districts]))
else:
    print("no importation pressure identified for this district-week")

### What to do about it (shortcoming #12)

In [ ]:
for i, rec in enumerate(prediction.recommendations, 1):
    print(f"{i}. {textwrap.fill(rec.action, 96, subsequent_indent='   ')}")
    print(f"   within {rec.timeframe_days} days | owner: {rec.responsible}")
    if rec.quantity:
        print(f"   quantity: {rec.quantity}")
    print()

## 7. Do the intervals actually cover?

A "95% interval" that covers 37% of outcomes is worse than no interval, because
it invites false confidence in a resource decision. This is measured, not assumed.

In [ ]:
from src.evaluation.calibration import interval_calibration
from src.evaluation.metrics import interval_metrics, regression_metrics

# Score every district on the weeks the model can be checked against.
rows = []
for district in matrix.districts:
    model = module.model_for(district)
    local = matrix.for_district(district).dropna_rows()
    if local.X.empty:
        continue
    point = np.clip(model.predict(local.X), 0, None)
    lower, upper = module.interval_bounds(model, point)
    rows.append(pd.DataFrame({
        "actual": local.y.to_numpy(dtype=float),
        "predicted": point, "lower": lower, "upper": upper,
    }))

scored = pd.concat(rows, ignore_index=True)
print("Point accuracy (in-sample - see notebook 04 for the honest out-of-sample view):")
display(pd.Series(regression_metrics(scored.actual, scored.predicted)).round(3).to_frame("value"))

calibration = interval_calibration(scored.actual, scored.lower, scored.upper)
print("\nInterval calibration:")
display(pd.Series(calibration.summary()).to_frame("value"))

## 8. Alerts

`detect_outbreak` compares each forecast to the disease's configured thresholds
and emits structured alerts. Nothing is raised below the `low` threshold — a
system that alerts on everything is ignored.

In [ ]:
predictions = module.predict_all(matrix, panel=panel)
alerts = module.detect_outbreak(predictions)

print(f"{len(predictions)} forecasts -> {len(alerts)} alert(s)\n")
if alerts:
    display(pd.DataFrame([{
        "district": a.district, "week": a.target_week, "level": a.risk_level,
        "cases": round(a.predicted_cases), "per_1000": round(a.predicted_incidence_per_1000, 3),
        "threshold": a.threshold_crossed, "low_confidence": a.low_data_confidence,
        "actions": len(a.recommendations),
    } for a in alerts]))
else:
    thresholds = module.config.alerts
    print(f"No district crossed the 'low' threshold of {thresholds.low} per 1,000 this week.")
    print("Forecast incidence by district:")
    display(pd.Series({
        p.district: round(module.incidence_per_1000(p.predicted_cases, p.district), 4)
        for p in predictions
    }).sort_values(ascending=False).to_frame("per_1000"))

## 9. Persist

Models are saved with a JSON sidecar recording the backend, training window and
version, so any forecast can be traced back to the code and data that produced it.

In [ ]:
path = module.save()
print(f"saved to {path}")
print(f"metadata: {path.with_suffix('.json')}\n")
print(path.with_suffix(".json").read_text())

reloaded = build_module(DISEASE, region=REGION)
print(f"\nreload check: {reloaded.load(path)} - {len(reloaded.models)} model(s) restored")

## 10. Distil an edge model

For a district node without the resources to run the full pipeline,
`LightweightModel` distils the trained model into ~10 coefficients. It reports
its own fidelity against the teacher rather than degrading silently.

In [ ]:
from offline.lightweight_model import LightweightModel

light = LightweightModel.distil(pooled, background, n_features=10,
                                provenance=matrix.provenance)
display(pd.Series(light.describe()).to_frame("value"))

print("\nCoefficients an epidemiologist can read by hand:")
display(pd.DataFrame({
    "feature": light.features,
    "coefficient": np.round(light.coefficients, 3),
    "mechanism": [light.feature_notes.get(f, "")[:55] for f in light.features],
}))

row = background.iloc[-1]
print(f"\nfull model : {pooled.predict(row.to_frame().T)[0]:,.1f} cases")
print(f"edge model : {light.predict(row.to_frame().T)[0]:,.1f} cases")

## Next

* **`04_model_comparison.ipynb`** — walk-forward validation, the three naive
  baselines every model must beat, and backend/ensemble comparison.
* **`05_spatial_validation.ipynb`** — does the disease spread where we said it would?
* **`06_drift_and_retraining.ipynb`** — the Google Flu Trends failure mode, caught.